In [129]:
import pandas as pd
import geopandas as gpd
import numpy as np
import pyogrio
import os

In [130]:
FOLDER = r"C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara"
OD_FOLDER = r"Insumos IMEPLAN\Matriz Origen Destino"
RED_FOLDER = r"Red Base GDL\RedBase Final\version base para rezonificacion"
ZONES_FOLDER = r"Visum Objects\Zones\zonas finales"

### Read CSV OD Original (all periods)

In [132]:
zones = gpd.read_file(os.path.join(FOLDER, ZONES_FOLDER, "zonas_2212.shp"))
od = pd.read_excel(os.path.join(FOLDER, OD_FOLDER, "matriz_origen_destino_agebs_por_hora_por_modo.xlsx"))

# Keep only public transport trips
od_public_transit = od[od["Tipo de transporte"] == "Transporte Público"].copy()

# Clear and replace Aeropuerto
# Patch: tratar 99999000A como el AGEB 14097059A
od_public_transit["Origen"] = (
    od_public_transit["Origen"]
    .astype(str)
    .str.strip()
    #.replace("99999000A", "14097059A")
)

od_public_transit["Destino"] = (
    od_public_transit["Destino"]
    .astype(str)
    .str.strip()
    #.replace("99999000A", "14097059A")
)

zones["clave_ageb"] = zones["clave_ageb"].astype(str).str.strip()

ageb_to_visum_zone = dict(zip(zones["clave_ageb"], zones["id_visum"]))

od_public_transit["Origen_visum"] = od_public_transit["Origen"].map(ageb_to_visum_zone)
od_public_transit["Destino_visum"] = od_public_transit["Destino"].map(ageb_to_visum_zone)

# Matriz 24h public transport trips
od_put_24h = od_public_transit.groupby(["Origen_visum", "Destino_visum"])["Sum of Ponderador"].sum().reset_index()
print(f"Total public transport trips: {od_put_24h['Sum of Ponderador'].sum():,}")
print(f"Unique OD pairs: {len(od_put_24h):,}")
print(f"Total nans in Origen_visum: {od_put_24h['Origen_visum'].isna().sum():,}")
print(f"Total nans in Destino_visum: {od_put_24h['Destino_visum'].isna().sum():,}")

Total public transport trips: 2,544,369
Unique OD pairs: 25,584
Total nans in Origen_visum: 0
Total nans in Destino_visum: 0


### Insert all day put matrix to Visum

In [133]:
import win32com.client as com

#Red base GDL (con 2,203 zonas)
red_base = os.path.join(FOLDER, RED_FOLDER, "RedFinal - rezonificacion - copia.ver")
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(red_base)
C = com.constants

In [134]:
zone_values = Visum.Net.Zones.GetMultiAttValues("No")

zone_nos = np.array(
    [int(value) for _, value in zone_values],
    dtype=np.int64
)

zone_to_index = {
    zone_no: index
    for index, zone_no in enumerate(zone_nos)
}

n_zones = len(zone_nos)
demand_array = np.zeros((n_zones, n_zones), dtype=np.float64)

origin_indices = (
    od_put_24h["Origen_visum"]
    .map(zone_to_index)
    .to_numpy(dtype=np.int64)
)

destination_indices = (
    od_put_24h["Destino_visum"]
    .map(zone_to_index)
    .to_numpy(dtype=np.int64)
)

# Numero de viajes
demand_values = (
    od_put_24h["Sum of Ponderador"]
    .to_numpy(dtype=np.float64)
)
# Llenar el array con los viajes reales
np.add.at(
    demand_array,
    (origin_indices, destination_indices),
    demand_values
)

visum_matrix_pf = Visum.Net.Matrices.ItemByKey(100)
visum_matrix_pf.SetValues(demand_array, Add=False)

### After PuT Assignment check PuT OD Paths

In [128]:
put_paths = pd.read_csv(os.path.join(FOLDER, RED_FOLDER, "put_od_paths_assigned.csv"))
put_paths = put_paths.rename(columns={"ORIGZONENO": "Origen_visum", "DESTZONENO": "Destino_visum", "ODTRIPSTOTAL": "Trips Visum"})
put_paths = put_paths[["Origen_visum", "Destino_visum", "Trips Visum"]]
put_paths = put_paths.groupby(["Origen_visum", "Destino_visum"])["Trips Visum"].sum().reset_index()
print(f"Total OD pairs in put_paths: {len(put_paths):,}")
print(f"{(len(put_paths)/len(od_put_24h)*100):.2f}% of demand assigned")
print(f"{len(od_put_24h) - len(put_paths):,} OD pairs unassigned")

KeyError: "['Trips Visum'] not in index"

In [126]:
comparison = od_put_24h.merge(
    put_paths,
    on=["Origen_visum", "Destino_visum"],
    how="left",
    indicator=True
)

missing_pairs = comparison[
    comparison["_merge"] == "left_only"
].copy()

missing_pairs = missing_pairs[missing_pairs["Origen_visum"] != missing_pairs["Destino_visum"]]
print(f"Total unassigned interzonal OD pairs: {len(missing_pairs):,}")


Total unassigned interzonal OD pairs: 0


In [127]:
missing_pairs.to_csv(os.path.join(FOLDER, RED_FOLDER, "unassigned_od_pairs.csv"), index=False)